# Детекция ботов 

Я строю признаки для каждой куки и сравниваю модели по **Precision при Recall ≥ 70%**. На временной валидации улучшенный Random Forest даёт **0,7517**, baseline — **0,1062**. 

## Подход

1. Я оставляю события только внутри суточного окна куки, чтобы признаки были доступны к его окончанию.
2. Я считаю типы действий, разнообразие объявлений и запросов, интервалы, активность по минутам, признаки курсора, платформы и User-Agent.
3. Я сравниваю простой Random Forest с моделями на расширенных признаках. Для проверки беру 17–19 апреля, более поздние даты относительно обучения.
4. Я выбираю модель по метрике, обучаю её на всём train и сохраняю `cookie_id,score`.

## 1. Настройка и импорт бибилиотек

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

RANDOM_STATE = 42
DATA_DIR = Path("data")
OUTPUT = Path("submission.csv")
assert (DATA_DIR / "train.csv").exists()
assert (DATA_DIR / "test.csv").exists()
assert (DATA_DIR / "events.csv.gz").exists()
assert Path("metric.py").exists()

## 2. Загрузка

In [4]:
date_columns = ["cookie_created_at", "window_start_ts", "window_end_ts"]
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=date_columns)
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=date_columns)
events = pd.read_csv(DATA_DIR / "events.csv.gz", parse_dates=["event_ts"])

print("Размеры train, test, events:", train.shape, test.shape, events.shape)
print("Доля ботов в train:", round(train.target.mean(), 4))
print("Train:", train.window_start_ts.min().date(), "—", train.window_start_ts.max().date())
print("Test:", test.window_start_ts.min().date(), "—", test.window_start_ts.max().date())


Размеры train, test, events: (11091, 5) (4909, 4) (328905, 14)
Доля ботов в train: 0.0811
Train: 2026-04-06 — 2026-04-19
Test: 2026-04-20 — 2026-04-26


### Проверка структуры

Я проверяю уникальность куки, типы событий и пропуски перед расчётом признаков.

In [5]:
assert train.cookie_id.is_unique and test.cookie_id.is_unique
assert not set(train.cookie_id).intersection(test.cookie_id)
print('Типы событий:')
print(events.event_name.value_counts().to_string())
print('Пропуски:')
print(events.isna().mean().round(3).to_string())
print('Полные дубликаты событий:', int(events.duplicated().sum()))


Типы событий:
event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Пропуски:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
Полные дубликаты событий: 4863


## 3. Метрика

Я импортирую реализацию из `metric.py`, чтобы одинаковые оценки учитывались так же, как при проверке.

In [6]:
from metric import precision_at_recall

assert precision_at_recall([1, 1, 0, 0], [0.5] * 4) == 0.5

## 4. Признаки

Я сначала исключаю события вне окна, затем агрегирую действия по `cookie_id`.

In [7]:
EVENT_TYPES = (
    "item_view", "search_results_view", "photo_swipe", "favorite_add",
    "seller_page_view", "contact_phone_show", "captcha_shown", "login",
    "contact_chat_open", "contact_message_sent",
)

def event_features(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """Я агрегирую только события внутри окна наблюдения каждой куки."""
    assert meta.cookie_id.is_unique
    ev = events.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id", how="inner", validate="many_to_one",
    )
    in_window = ev.event_ts.ge(ev.window_start_ts) & ev.event_ts.lt(ev.window_end_ts)
    print(f"Я исключил события вне окон: {(~in_window).sum():,}")
    ev = ev.loc[in_window].copy()
    ev = ev.sort_values(["cookie_id", "event_ts"], kind="mergesort")

    # Я сохраняю повторы событий и считаю интервалы по времени, без опоры на порядок строк.
    ev["platform_norm"] = ev.platform.fillna("").str.lower().replace({
        "web": "web", "desktop": "desktop", "android": "android",
        "iphone": "ios", "ios": "ios",
    })
    ua = ev.user_agent.fillna("").str.lower()
    ev["ua_headless"] = ua.str.contains("headless|selenium|webdriver", regex=True).astype("int8")
    ev["ua_script"] = ua.str.contains("python|requests|curl|scrapy|httpclient", regex=True).astype("int8")
    ev["ua_linux"] = ua.str.contains("linux").astype("int8")
    ev["ua_mobile"] = ua.str.contains("android|iphone|ipad").astype("int8")
    ev["browser"] = np.select(
        [ua.str.contains("headlesschrome"), ua.str.contains("yabrowser"),
         ua.str.contains("firefox"), ua.str.contains("chrome")],
        ["headless", "yandex", "firefox", "chrome"], default="other",
    )
    ev["hour"] = ev.event_ts.dt.hour
    ev["minute"] = ev.event_ts.dt.floor("min")
    ev["elapsed_seconds"] = (ev.event_ts - ev.window_start_ts).dt.total_seconds()
    ev["gap_seconds"] = ev.groupby("cookie_id", sort=False).event_ts.diff().dt.total_seconds()
    ev["gap_zero"] = ev.gap_seconds.eq(0).astype("int8")
    ev["gap_short"] = ev.gap_seconds.between(0, 2, inclusive="both").astype("int8")
    ev["gap_long"] = ev.gap_seconds.gt(300).astype("int8")
    ev["is_night"] = ev.hour.between(0, 5).astype("int8")
    ev["query_length"] = ev.search_query.fillna("").str.len()
    ev["query_clean"] = ev.search_query.fillna("").str.lower().str.strip()
    ev["pointer_present"] = ev.pointer_x.notna() & ev.pointer_y.notna()

    grouped = ev.groupby("cookie_id", sort=False)
    f = grouped.agg(
        n_events=("event_ts", "size"),
        n_event_types=("event_name", "nunique"),
        item_nunique=("item_id", "nunique"),
        category_nunique=("item_category", "nunique"),
        location_nunique=("item_location", "nunique"),
        seller_type_nunique=("seller_type", "nunique"),
        query_nunique=("query_clean", lambda s: s[s.ne("")].nunique()),
        ua_nunique=("user_agent", "nunique"),
        browser_nunique=("browser", "nunique"),
        platform_nunique=("platform_norm", "nunique"),
        active_minutes=("minute", "nunique"),
        active_hours=("hour", "nunique"),
        first_elapsed=("elapsed_seconds", "min"),
        last_elapsed=("elapsed_seconds", "max"),
        gap_mean=("gap_seconds", "mean"),
        gap_median=("gap_seconds", "median"),
        gap_std=("gap_seconds", "std"),
        gap_min=("gap_seconds", "min"),
        gap_max=("gap_seconds", "max"),
        gap_zero_share=("gap_zero", "mean"),
        gap_short_share=("gap_short", "mean"),
        gap_long_share=("gap_long", "mean"),
        night_share=("is_night", "mean"),
        ua_headless_share=("ua_headless", "mean"),
        ua_script_share=("ua_script", "mean"),
        ua_linux_share=("ua_linux", "mean"),
        ua_mobile_share=("ua_mobile", "mean"),
        pointer_share=("pointer_present", "mean"),
        pointer_x_nunique=("pointer_x", "nunique"),
        pointer_y_nunique=("pointer_y", "nunique"),
        pointer_x_std=("pointer_x", "std"),
        pointer_y_std=("pointer_y", "std"),
        search_page_max=("search_page", "max"),
        search_page_mean=("search_page", "mean"),
        query_length_mean=("query_length", "mean"),
        query_length_max=("query_length", "max"),
    )
    f["gap_q10"] = grouped.gap_seconds.quantile(0.10)
    f["gap_q90"] = grouped.gap_seconds.quantile(0.90)
    f["span_seconds"] = f.last_elapsed - f.first_elapsed

    minute_counts = ev.groupby(["cookie_id", "minute"]).size()
    f["peak_events_per_minute"] = minute_counts.groupby(level=0).max()
    f["mean_events_per_active_minute"] = f.n_events / f.active_minutes
    f["item_repeat_ratio"] = 1 - f.item_nunique / f.n_events
    f["query_repeat_ratio"] = 1 - f.query_nunique / f.n_events
    f["item_diversity_per_view"] = f.item_nunique / grouped.item_id.count().clip(lower=1)

    for field, values, prefix in [
        ("event_name", EVENT_TYPES, "event"),
        ("platform_norm", ("web", "desktop", "android", "ios"), "platform"),
        ("browser", ("headless", "yandex", "firefox", "chrome", "other"), "browser"),
    ]:
        counts = pd.crosstab(ev.cookie_id, ev[field]).reindex(
            index=f.index, columns=values, fill_value=0,
        )
        for value in values:
            column = f"{prefix}_{value}"
            f[column] = counts[value]
            f[column + "_share"] = counts[value] / f.n_events

    # Я сохраняю все куки: пропуск агрегата кодирую как -1, настоящий счётчик — как 0.
    out = meta[["cookie_id", "cookie_created_at", "window_start_ts"]].copy()
    out["cookie_age_days"] = (
        out.window_start_ts - out.cookie_created_at
    ).dt.total_seconds() / 86400
    out = out.drop(columns=["cookie_created_at", "window_start_ts"])
    out = out.merge(f.reset_index(), on="cookie_id", how="left", validate="one_to_one")
    count_columns = ["n_events", "item_nunique", "active_minutes", "active_hours"]
    out[count_columns] = out[count_columns].fillna(0)
    return out.fillna(-1)

### Матрица признаков

Я присоединяю агрегаты к train и test, сохраняя порядок куки.

In [8]:
all_meta = pd.concat([train, test], ignore_index=True)
features = event_features(events, all_meta)
data_train = train[['cookie_id']].merge(features, on='cookie_id', validate='one_to_one')
data_test = test[['cookie_id']].merge(features, on='cookie_id', validate='one_to_one')
feature_columns = data_train.columns.drop('cookie_id').tolist()
baseline_columns = ['n_events', 'item_nunique']
assert data_train.cookie_id.equals(train.cookie_id)
assert data_test.cookie_id.equals(test.cookie_id)
print('Число признаков:', len(feature_columns))
print('Пропуски после обработки:', int(data_train[feature_columns].isna().sum().sum()))


Я исключил события вне окон: 40,779
Число признаков: 83
Пропуски после обработки: 0


## 5. Валидация

Я откладываю последние три дня train и сравниваю модели по основной метрике.

In [10]:
def evaluate(name, model, train_x, valid_x, y_train, y_valid):
    model.fit(train_x, y_train)
    p = model.predict_proba(valid_x)[:, 1]
    result = {
        "model": name,
        "P@R≥0.70": precision_at_recall(y_valid, p),
        "PR-AUC": average_precision_score(y_valid, p),
        "ROC-AUC": roc_auc_score(y_valid, p),
    }
    print(" | ".join(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}" for k, v in result.items()))
    return result

is_valid = train.window_start_ts.ge('2026-04-17').to_numpy()
x_train = data_train.loc[~is_valid, feature_columns]
x_valid = data_train.loc[is_valid, feature_columns]
y_train = train.loc[~is_valid, 'target'].to_numpy()
y_valid = train.loc[is_valid, 'target'].to_numpy()
print(f'Train: {len(y_train):,}; validation: {len(y_valid):,}; доля ботов: {y_valid.mean():.4f}')


Train: 9,140; validation: 1,951; доля ботов: 0.0820


### Сравнение

Я использую два признака для baseline и все признаки поведения для улучшенных моделей.

In [11]:
baseline = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=1,
)
forest = RandomForestClassifier(
    n_estimators=400, min_samples_leaf=2, max_features=0.7,
    random_state=RANDOM_STATE, n_jobs=1,
)
boosting = HistGradientBoostingClassifier(
    max_iter=250, max_leaf_nodes=15, min_samples_leaf=25,
    l2_regularization=2.0, learning_rate=0.05, random_state=RANDOM_STATE,
)
baseline_result = evaluate('RF, 2 baseline features', baseline,
    x_train[baseline_columns], x_valid[baseline_columns], y_train, y_valid)
improved_results = [
    evaluate('RF, behavior features', forest, x_train, x_valid, y_train, y_valid),
    evaluate('HistGradientBoosting, behavior features', boosting,
             x_train, x_valid, y_train, y_valid),
]
print(pd.DataFrame([baseline_result, *improved_results]).round(4).to_string(index=False))

model: RF, 2 baseline features | P@R≥0.70: 0.1062 | PR-AUC: 0.1804 | ROC-AUC: 0.6081
model: RF, behavior features | P@R≥0.70: 0.7517 | PR-AUC: 0.7373 | ROC-AUC: 0.9150
model: HistGradientBoosting, behavior features | P@R≥0.70: 0.7417 | PR-AUC: 0.7599 | ROC-AUC: 0.9264
                                  model  P@R≥0.70  PR-AUC  ROC-AUC
                RF, 2 baseline features    0.1062  0.1804   0.6081
                  RF, behavior features    0.7517  0.7373   0.9150
HistGradientBoosting, behavior features    0.7417  0.7599   0.9264


## 6. Предсказания

Я переобучаю выбранную модель на всём train и записываю оценки без фиксированного порога.

In [12]:
selected_name = max(improved_results, key=lambda result: result['P@R≥0.70'])['model']
selected_model = forest if selected_name.startswith('RF') else boosting
print('Выбранная модель:', selected_name)
selected_model.fit(data_train[feature_columns], train.target)
submission = pd.DataFrame({
    'cookie_id': test.cookie_id,
    'score': selected_model.predict_proba(data_test[feature_columns])[:, 1],
})
assert list(submission.columns) == ['cookie_id', 'score']
assert len(submission) == len(test) and submission.cookie_id.is_unique
assert submission.cookie_id.equals(test.cookie_id)
assert submission.score.notna().all() and submission.score.between(0, 1).all()
submission.to_csv(OUTPUT, index=False)
print(f'{OUTPUT}: {len(submission):,} строк')
print(submission.head(5).to_string(index=False))


Выбранная модель: RF, behavior features
submission.csv: 4,909 строк
          cookie_id    score
ck_315fb710a0e371e7 0.001071
ck_a76ee3b3e3e522fd 0.225667
ck_94c9a4d382689e82 0.015417
ck_8eaf9509ad9462a0 0.001667
ck_9a88a5a989cb5bc6 0.008542


## Вывод

Я получила P@R≥0,70 **0,7517** у улучшенного Random Forest против **0,1062** у baseline. В `submission.csv` я сохраняю 4 909 строк в порядке `test.csv`.
